In [ ]:
import requests
from selenium import webdriver
from selenium.webdriver.support.ui import WebDriverWait
import pandas as pd
import lxml.etree as et
import json
import datetime
import time

In [ ]:
# record script start time
sstime = time.time()
# define paths and urls
root_url = 'https://archaeologydataservice.ac.uk/archives/view/eab_eh_2004/'
query_url = 'https://archaeologydataservice.ac.uk/archives/view/eab_eh_2004/query.cfm'
driver_path = 'chromedriver'  # update this to your local chromedriver path, or leave as-is if chromedriver is on your PATH
# define HTML parser
htmlparser = et.HTMLParser()
# create container for scraped results
output = []
# create log
results_log = []

In [ ]:
# load selenium driver
driver = webdriver.Chrome(executable_path=driver_path)
# set wait time
driver.implicitly_wait(10) # in seconds
# get query landing page
driver.get(query_url)
# run an empty query to get all results
driver.find_element_by_name('query').click()

In [ ]:
#loop through results pages
is_done = 0
page_no = 1
while is_done == 0:
    # record start time
    stime = time.time()
    # get the HTML from the current results page
    page_source = driver.page_source
    # pass it to lxml to parse
    xtree = et.XML(page_source, htmlparser)
    # create a list with all the results
    results = xtree.findall('.//div[@class="greybkgd"]/strong/a')
    # create a list with all the links to the full records
    result_links = []
    for atag in results:
        result_links.append(atag.attrib['href'])
    # scrape each individual record
    for link in result_links:
        # build the url
        record_url = root_url + link
        # pull the page
        request = requests.get(record_url)
        # pass it to lxml to parse
        response = et.XML(request.content, htmlparser)
        # create a dictionary to hold the values
        record_dict = {}
        # get the record's title
        if response.find('.//div[@id="archive"]/h4[1]') is not None:
            record_dict['Name'] = response.find('.//div[@id="archive"]/h4[1]').text
        else:
            record_dict['Name'] = 'N/A'
        # get the metadata table
        meta_rows = response.findall('.//div[@id="archive"]/table[1]/tr')
        # extract the labels and values from the metadata table
        for row in meta_rows:
            record_dict[str(row.find('.//th').text).strip()] = str(row.find('.//td').text).strip()
        # create list to hold specialist reports
        spec_reports = []
        # get the specialist reports table
        spec_rows = response.findall('.//div[@id="archive"]/table[2]/tr')
        if spec_rows:
            # extract the metadata for each report and add it to the record's dictionary
            repo_dict = {}
            for n, row in enumerate(spec_rows):
                if len(row.findall('.//th')) == 0:
                    if repo_dict:
                        spec_reports.append(repo_dict)
                    repo_dict = {'Report': row.find('.//td/strong').text.strip()}  
                else:
                    val = [i.strip() for i in row.find('.//td').itertext() if i.strip() != '']
                    if len(val) == 1:
                        val = val[0]
                    repo_dict[row.find('.//th').text.strip()] = val

                if n+1 == len(spec_rows):
                    spec_reports.append(repo_dict)
        # add reports list to record's dictionary
        record_dict['Specialist Reports'] = spec_reports
        # add the record's dictionary to the results
        output.append(record_dict)
    # record end time
    etime = time.time()
    # update log
    time_delta = datetime.timedelta(seconds=etime - stime)
    minutes, seconds = divmod(time_delta.seconds, 60)
    time_str = '{} m {} s'.format(str(minutes), str(seconds))
    results_log.append('page ' + str(page_no) + ' completed in ' + time_str)
    # move to next page
    if len(driver.find_elements_by_xpath('//a[@title="Go forward 10 records"]')) != 0:
        page_no += 1
        driver.find_element_by_xpath('//a[@title="Go forward 10 records"]').click()
    else:
        is_done = 1
        
#close browser window
driver.close()
# save results to disk as JSON
with open('EAB_database.json', 'w') as db_file:
    json.dump(output, db_file)
# record script end time
setime = time.time()
# update log
time_delta = datetime.timedelta(seconds=setime - sstime)
minutes, seconds = divmod(time_delta.seconds, 60)
time_str = '{} m {} s'.format(str(minutes), str(seconds))
results_log.append('Scraping completed in ' + time_str)
# write log to disk
with open('log.txt', 'w') as log_file:
    for entry in results_log:
        log_file.write(entry + '\n')